### Build a Question Answering Application on a Graph DB

In [19]:
from dotenv import load_dotenv
import os
from langchain_neo4j import Neo4jGraph
from langchain_openai import ChatOpenAI
from langchain_neo4j import GraphCypherQAChain

In [ ]:
load_dotenv()

neo4j_password = os.getenv("NEO4J_PASSWORD")

In [51]:
graph = Neo4jGraph(
    url="bolt://localhost:7687",
    username="neo4j",
    password=neo4j_password,
    database="myDB",
    refresh_schema=False  # keep this until APOC is confirmed
)

result = graph.query("RETURN 1 AS test")
print(result) 

[{'test': 1}]


In [52]:
movie_query = """
LOAD CSV WITH HEADERS FROM 
'https://raw.githubusercontent.com/tomasonjo/blog-datasets/main/movies/movies_small.csv' AS row
MERGE (m:Movie {id: row.movieId})
SET 
    m.released = date(row.released),
    m.title = row.title,
    m.imdbRating = toFloat(row.imdbRating)
FOREACH (director IN split(row.director, '|') |
    MERGE (p:Person {name: trim(director)})
    MERGE (p)-[:DIRECTED]->(m)
)
FOREACH (actor IN split(row.actors, '|') |
    MERGE (p:Person {name: trim(actor)})
    MERGE (p)-[:ACTED_IN]->(m)
)
FOREACH (genre IN split(row.genres, '|') |
    MERGE (g:Genre {name: trim(genre)})
    MERGE (m)-[:IN_GENRE]->(g)
)
"""

# Run it
graph.query(movie_query)
print("✅ Data loaded!")

✅ Data loaded!


In [33]:
graph.refresh_schema()
print("✅ Schema refreshed!")

✅ Schema refreshed!


In [53]:
print(graph.schema)

In [35]:
llm = ChatOpenAI(model="gpt-4o", temperature=0)

In [36]:
chain = GraphCypherQAChain.from_llm(
    llm=llm,
    graph=graph,
    verbose=True,
    allow_dangerous_requests=True,
    cypher_llm_kwargs={"temperature": 0},  # deterministic queries
)

In [37]:
response = chain.invoke({"query": "Who is the director of the movie Casino?"})



> Entering new GraphCypherQAChain chain...
Generated Cypher:
cypher
MATCH (p:Person)-[:DIRECTED]->(m:Movie {title: "Casino"})
RETURN p.name

Full Context:
[{'p.name': 'Martin Scorsese'}]

> Finished chain.


In [38]:
print("Director of Casino:", response['result'])

Director of Casino: Martin Scorsese is the director of the movie Casino.


In [24]:
query = """Who are the actors in the movie with the highest IMDb rating?"""
response = chain.invoke({"query": query})
print("Actors in the highest rated movie:", response['result'])



> Entering new GraphCypherQAChain chain...
Generated Cypher:
cypher
MATCH (m:Movie)<-[:ACTED_IN]-(p:Person)
WITH m, p
ORDER BY m.imdbRating DESC
LIMIT 1
RETURN p.name

Full Context:
[{'p.name': 'Tim Robbins'}]

> Finished chain.
Actors in the highest rated movie: Tim Robbins is an actor in the movie with the highest IMDb rating.


### Prompting Strategies with Graph DB

In [54]:
from langchain_core.prompts import FewShotPromptTemplate, PromptTemplate

In [72]:
examples = [
    {
        "question": "How many artists are there?",
        "query": "MATCH (a:Person)-[:ACTED_IN]->(:Movie) RETURN count(DISTINCT a)",
    },
    {
        "question": "Which actors played in the movie Casino?",
        "query": "MATCH (m:Movie {{title: 'Casino'}})<-[:ACTED_IN]-(a) RETURN a.name",
    },
    {
        "question": "How many movies has Tom Hanks acted in?",
        "query": "MATCH (a:Person {{name: 'Tom Hanks'}})-[:ACTED_IN]->(m:Movie) RETURN count(m)",
    },
    {
        "question": "List all the genres of the movie Schindler's List",
        "query": "MATCH (m:Movie {{title: \"Schindler's List\"}})-[:IN_GENRE]->(g:Genre) RETURN g.name",
    },
    {
        "question": "Which actors have worked in movies from both the comedy and action genres?",
        "query": "MATCH (a:Person)-[:ACTED_IN]->(:Movie)-[:IN_GENRE]->(g1:Genre), (a)-[:ACTED_IN]->(:Movie)-[:IN_GENRE]->(g2:Genre) WHERE g1.name = 'Comedy' AND g2.name = 'Action' RETURN DISTINCT a.name",
    },
    {
        "question": "Which directors have made movies with at least three different actors named 'John'?",
        "query": "MATCH (d:Person)-[:DIRECTED]->(m:Movie)<-[:ACTED_IN]-(a:Person) WHERE a.name STARTS WITH 'John' WITH d, COUNT(DISTINCT a) AS JohnsCount WHERE JohnsCount >= 3 RETURN d.name",
    },
    {
        "question": "Identify movies where directors also played a role in the film.",
        "query": "MATCH (p:Person)-[:DIRECTED]->(m:Movie), (p)-[:ACTED_IN]->(m) RETURN m.title, p.name",
    },
    {
        "question": "Find the actor with the highest number of movies in the database.",
        "query": "MATCH (a:Person)-[:ACTED_IN]->(m:Movie) RETURN a.name, COUNT(m) AS movieCount ORDER BY movieCount DESC LIMIT 1",
    },
]

In [73]:
example_prompt = PromptTemplate(
    input_variables=["question", "query"],
    template="Question: {question}\nCypher Query: {query}"
)

prompt = FewShotPromptTemplate(
    input_variables=["schema", "question"],  # 👈 must have both
    example_prompt=example_prompt,
    examples=examples,
    prefix="""You are an expert Neo4j developer. 
Given the schema below, use the examples to generate a Cypher query.

Schema:
{schema}
""",
    suffix="Question: {question}\nCypher Query:",  # 👈 {question} not {query}
)

In [77]:
chain = GraphCypherQAChain.from_llm(
    llm=llm,
    graph=graph,
    verbose=True,
    allow_dangerous_requests=True,
    return_intermediate_steps=True, 
    cypher_llm_kwargs={
        "temperature": 0,
        "prompt": prompt
    }
)

In [75]:
# Test 1 - Simple question
result = chain.invoke("Who directed The Matrix?")
print(result["result"])



> Entering new GraphCypherQAChain chain...
Generated Cypher:
cypher
MATCH (d:Person)-[:DIRECTED]->(m:Movie {title: 'The Matrix'}) RETURN d.name

Full Context:
[]

> Finished chain.
I don't know the answer.


In [78]:
result = chain.invoke("Who directed The Matrix?")

print("🔍 Generated Cypher:")
print(result["intermediate_steps"][0]["query"])

print("\n💬 Answer:")
print(result["result"])



> Entering new GraphCypherQAChain chain...
Generated Cypher:
cypher
MATCH (d:Person)-[:DIRECTED]->(m:Movie {title: 'The Matrix'}) RETURN d.name

Full Context:
[]

> Finished chain.
🔍 Generated Cypher:
cypher
MATCH (d:Person)-[:DIRECTED]->(m:Movie {title: 'The Matrix'}) RETURN d.name


💬 Answer:
I don't know the answer.
